# Retrieval: BM25 and lexical refinements

One workflow from the validated Step 5 chunks to source-backed retrieval. Builds both indexes, compares the original and enhanced search, and displays evidence. No separate Step 6 notebook is required. Steps 3/4 remain skipped; no embeddings or model calls.

## Save configuration

Baseline database/manifest/checks in BASELINE_OUTPUT_ROOT. Enhanced database package, comparison, example evidence, ablations and summary in OUTPUT_ROOT.

Set the output directory below before running. Each export creates a new run subdirectory beneath it and returns its actual path. Saving happens when the export/run cell executes; the script receives this directory explicitly. Existing files are not overwritten. Changing output paths does not change downstream input discovery automatically; pass explicit bundle/index paths when using custom locations.

In [1]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'pyproject.toml').exists())
OUTPUT_ROOT = ROOT / 'artifacts' / '03_retrieval_enhanced'
BASELINE_OUTPUT_ROOT = ROOT / 'artifacts' / '03_retrieval_baseline'
print('Baseline save directory:', BASELINE_OUTPUT_ROOT.resolve())
print('Save directory:', OUTPUT_ROOT.resolve())

Baseline save directory: C:\Users\PK\Desktop\projects\mobile_rag\artifacts\03_retrieval_baseline
Save directory: C:\Users\PK\Desktop\projects\mobile_rag\artifacts\03_retrieval_enhanced


## 1. Build and compare

The runner selects the latest validated Step 5 bundle. Pass `run(bundle=Path(...))` to select one explicitly. It compares all supplied question strings without using reference answers. Each execution creates fresh artifact directories.

In [2]:
from pathlib import Path
import json
from run_step import run

ENABLE_BM25 = True
ENABLE_EMBEDDINGS = True
out = run(baseline_output_root=BASELINE_OUTPUT_ROOT, output_root=OUTPUT_ROOT, enable_bm25=ENABLE_BM25, enable_embeddings=ENABLE_EMBEDDINGS)

{
  "output": "C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\03_retrieval_enhanced\\20260913T120112863801Z",
  "queries": 204,
  "changed_top5": 199,
  "clinical_accuracy": "not_measured",
  "read_only_hashes_unchanged": true,
  "database_bytes": 68886528,
  "timings_ms": {
    "baseline": {
      "median": 34.56495003774762,
      "p95": 49.92579994723201
    },
    "enhanced": {
      "median": 153.33489998010918,
      "p95": 205.96020005177706
    }
  }
}


## 2. Inspect cost and ranking changes

Desktop timing and changed rankings do not establish clinical accuracy or phone performance.

In [3]:
summary = json.loads((out / 'summary.json').read_text())
summary

{'changed_top5': 199,
 'clinical_accuracy': 'not_measured',
 'database_bytes': 68886528,
 'queries': 204,
 'read_only_hashes_unchanged': True,
 'timings_ms': {'baseline': {'median': 34.56495003774762,
   'p95': 49.92579994723201},
  'enhanced': {'median': 153.33489998010918, 'p95': 205.96020005177706}}}

## 3. Inspect a source-backed result

Change the question and rerun this cell without rebuilding. Full source chunks and citation metadata are retained; scores express retrieval rank, not medical confidence.

In [6]:
from mobile_rag.retrieval_hybrid import HybridRetriever, RetrievalConfig

question = 'What should I remember about bubble CPAP?'
with HybridRetriever(out, RetrievalConfig(ENABLE_BM25, ENABLE_EMBEDDINGS)) as retriever:
    result = retriever.search(question)
    context = retriever.expand(result)
print('Status:', result['status'])
for hit in result['hits']:
    print('Rank:', hit['rank'], 'Branches:', hit['branch_ranks'])
    print('Citation:', hit['citation'])
    print(hit['chunk']['retrieval_text'])
    print()

Status: ok
Rank: 1 Branches: {'baseline': 7, 'heading': 7, 'focused': 1, 'phrase': 1, 'proximity': 1, 'aliases': 7, 'passages': 1}
Citation: {'chunk_id': 'chunk_4130f4c2a7a07a1f8e0b', 'citation_target_id': 'citation_6352a9c184321b74a08c', 'declared_pages': [4], 'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1', 'pdf_coordinate_status': 'unavailable', 'source_passage_ids': ['passage_dd26ed1bd8978f0122fc']}
# Guidelines for Use of Bubble-CPAP Concentrators


### Setting up the bubble-CPAP concentrator


Follow the steps below:


Rank: 2 Branches: {'baseline': 5, 'heading': 8, 'focused': 2, 'phrase': 2, 'proximity': 2, 'aliases': 8, 'passages': 3}
Citation: {'chunk_id': 'chunk_c6417a05d9b8067a2fd2', 'citation_target_id': 'citation_8b6ee256531149a5cef6', 'declared_pages': [5], 'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1', 'pdf_coordinate_status': 'unavailable', 'source_passage_ids': ['passage_f040dd56f8f6954078bb'

## 4. Inspect feature ablations

These saved rankings disable one feature at a time for the demonstration query. They are engineering evidence, not a clinical accuracy score.

In [7]:
json.loads((out / 'sample_ablations.json').read_text())

{'aliases': ['chunk_4130f4c2a7a07a1f8e0b',
  'chunk_c6417a05d9b8067a2fd2',
  'chunk_3dc64ac39513658e687c',
  'chunk_d58e8f3db498d1ba7230',
  'chunk_1416ee61798ba3a3e4d3'],
 'focused': ['chunk_4130f4c2a7a07a1f8e0b',
  'chunk_c6417a05d9b8067a2fd2',
  'chunk_3dc64ac39513658e687c',
  'chunk_1416ee61798ba3a3e4d3',
  'chunk_d58e8f3db498d1ba7230'],
 'heading': ['chunk_4130f4c2a7a07a1f8e0b',
  'chunk_c6417a05d9b8067a2fd2',
  'chunk_3dc64ac39513658e687c',
  'chunk_d58e8f3db498d1ba7230',
  'chunk_1416ee61798ba3a3e4d3'],
 'passages': ['chunk_4130f4c2a7a07a1f8e0b',
  'chunk_c6417a05d9b8067a2fd2',
  'chunk_3dc64ac39513658e687c',
  'chunk_d58e8f3db498d1ba7230',
  'chunk_5fafebb4b9edb42b6847'],
 'phrase': ['chunk_4130f4c2a7a07a1f8e0b',
  'chunk_c6417a05d9b8067a2fd2',
  'chunk_1416ee61798ba3a3e4d3',
  'chunk_3dc64ac39513658e687c',
  'chunk_d58e8f3db498d1ba7230'],
 'proximity': ['chunk_4130f4c2a7a07a1f8e0b',
  'chunk_c6417a05d9b8067a2fd2',
  'chunk_1416ee61798ba3a3e4d3',
  'chunk_3dc64ac39513658e687c',